In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
rcParams['font.sans-serif'] = ['SimHei']
rcParams['axes.unicode_minus'] = False
import seaborn as sns
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
#将数据输出结果导出
import os

In [ ]:
df=pd.read_csv('data/clean_2019-Nov.csv')
'''-------------商品关联规则分析:挖掘用户订单中经常一起购买的商品组合，输出支持度、置信度、提升度高的关联规则，用于凑单推荐、详情页关联推荐、购物车推荐，提升客单价。-------------'''

# 1. 数据预处理：按用户+日期作为一个购物篮，提取购买的商品
buy_df = df[df['event_type'] == 'purchase']

# 预过滤：保留销量前500的商品，减少one-hot维度，大幅提升运算速度
# 数据量小可注释掉本行
top_products = buy_df['product_id'].value_counts().head(200).index
buy_df = buy_df[buy_df['product_id'].isin(top_products)]

baskets = buy_df.groupby(['user_id', 'date'])['product_id'].apply(list).tolist()

# 2. 转成one-hot编码格式
te = TransactionEncoder()
te_array = te.fit_transform(baskets)
basket_df = pd.DataFrame(te_array, columns=te.columns_)
#调低最小支持度阈值，比如调到 0.005（0.5%），同时可以配合之前的预过滤长尾商品逻辑：
frequent_itemsets = apriori(basket_df, min_support=0.005, use_colnames=True)
# 3. 挖掘频繁项集（最小支持度0.02）
frequent_itemsets = apriori(basket_df, min_support=0.002, use_colnames=True)

# 4. 生成关联规则（按提升度筛选）
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.2)

# 5. 按提升度排序取Top10
top_rules = rules.sort_values('lift', ascending=False).head(10)[
    ['antecedents', 'consequents', 'support', 'confidence', 'lift']
]
print("总订单数：", len(baskets))
print("销量最高商品的订单数：", buy_df['product_id'].value_counts().iloc[0])

print(" 商品关联规则Top10（按提升度排序）:")
print("指标说明：")
print("  support(支持度)：该组合在所有订单中出现的比例")
print("  confidence(置信度)：买了A的用户中，同时买B的比例")
print("  lift(提升度)：关联规则相对于随机购买的提升倍数，>1才有效")
print(top_rules.round(4).to_string(index=False))
# ====================== 保存数据用于后续可视化输出 ======================
top_rules.to_csv('output/data/association_rules.csv',
                 index=False, encoding='utf-8-sig')